# PIE_Bench++ 评测

这个 notebook 会直接调用 `evaluate_metrics.py` 跑完整评测流程，并输出 `psnr`、`dino_dist`、`lpips`、`mse` 四个指标。

如果你本地已经有可用的 SD3 Diffusers 模型目录，把 `MODEL_KEY` 改成那个本地路径即可；否则默认使用 Hugging Face 的 `stabilityai/stable-diffusion-3-medium-diffusers`。


In [ ]:
from pathlib import Path
from argparse import Namespace
import json
import sys
import torch

REPO_ROOT = Path("/home/ljc/code/FlowAlign-main")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from evaluate_metrics import resolve_dataset_layout, evaluate_on_dataset

DATASET_PATH = REPO_ROOT / "PIE_Bench_pp"
OUTPUT_DIR = REPO_ROOT / "eval_results"
MODEL_KEY = "stabilityai/stable-diffusion-3-medium-diffusers"
METHOD = "flowalign"
IMG_SHAPE = 1024
CFG_SCALE = 13.5
NFE = 33
N_START = 0
SEED = 123
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

dataset_root, images_dir, edits_file = resolve_dataset_layout(DATASET_PATH)
print("dataset_root:", dataset_root)
print("images_dir:", images_dir)
print("edits_file:", edits_file)
print("device:", DEVICE)


In [ ]:
args = Namespace(
    dataset_path=str(DATASET_PATH),
    output_dir=str(OUTPUT_DIR),
    model="sd3",
    method=METHOD,
    model_key=MODEL_KEY,
    img_shape=IMG_SHAPE,
    cfg_scale=CFG_SCALE,
    NFE=NFE,
    n_start=N_START,
    seed=SEED,
    device=DEVICE,
)

results = evaluate_on_dataset(args)
print("\nmetric summary:")
print(json.dumps(results["metrics"], indent=2, ensure_ascii=False) if results else "No results returned")


In [ ]:
results_file = OUTPUT_DIR / "metrics.json"
if results_file.exists():
    print(results_file)
    print(results_file.read_text()[:2000])
else:
    print("metrics.json not found yet")
